In [13]:
!pip install rasterio geopandas shapely pysheds scikit-image -q

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [15]:
import os

INPUT_DIR  = "/content/drive/MyDrive/SVAMITVA_Data_DTM_MULTI/DTM_FILES_GENERATED_DL"
OUTPUT_DIR = "/content/drive/MyDrive/SVAMITVA_Drainage_Output"

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [16]:
dtm_files = [f for f in os.listdir(INPUT_DIR) if f.endswith(".tif")]

print(f"Found {len(dtm_files)} DTM files:")
for f in dtm_files:
    print(" -", f)

Found 6 DTM files:
 - 67169_5NKR_CHAKHIRASINGH_DTM.tif
 - Dhal_Hoshiarpur_31235_DTM.tif
 - KHAPRETA_510206_DTM.tif
 - DHUNDA_FATEHGARH SAHIB_32619_DTM.tif
 - 64334_2H (REFLIGHT)_POINT CLOUD_DTM.tif
 - DEVDI_POINT CLOUD (511671)_DTM.tif


In [17]:
import rasterio
import numpy as np
from pysheds.grid import Grid
import geopandas as gpd
from shapely.geometry import Point
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import binary_dilation

def process_dtm(dtm_path, output_prefix):
    print(f"\nProcessing: {os.path.basename(dtm_path)}")

    # Load DTM
    with rasterio.open(dtm_path) as src:
        dtm = src.read(1)
        transform = src.transform
        crs = src.crs

    dtm = np.where(np.isnan(dtm), np.nanmean(dtm), dtm)

    # Hydrology
    grid = Grid.from_raster(dtm_path)
    dem = grid.read_raster(dtm_path)

    pit_filled = grid.fill_pits(dem)
    flooded    = grid.fill_depressions(pit_filled)
    inflated   = grid.resolve_flats(flooded)

    flow_dir = grid.flowdir(inflated)
    acc      = grid.accumulation(flow_dir)

    # Drainage thresholds
    primary_thresh   = np.percentile(acc, 99)
    secondary_thresh = np.percentile(acc, 97)
    tertiary_thresh  = np.percentile(acc, 95)

    primary   = acc >= primary_thresh
    secondary = (acc >= secondary_thresh) & (acc < primary_thresh)
    tertiary  = (acc >= tertiary_thresh) & (acc < secondary_thresh)

    # Waterlogging
    gy, gx = np.gradient(dtm)
    slope  = np.sqrt(gx**2 + gy**2)

    low_slope = slope < np.percentile(slope, 25)
    high_acc  = acc > np.percentile(acc, 75)
    low_elev  = dtm < np.percentile(dtm, 30)

    waterlog = low_slope & high_acc & low_elev

    # ---- Convert to GeoDataFrame ----
    def raster_to_points(mask, label):
        rows, cols = np.where(mask)
        points = []

        for r, c in zip(rows, cols):
            x, y = rasterio.transform.xy(transform, r, c)
            points.append({"geometry": Point(x, y), "type": label})

        return gpd.GeoDataFrame(points, crs=crs)

    gdf = pd.concat([
        raster_to_points(primary, "Primary"),
        raster_to_points(secondary, "Secondary"),
        raster_to_points(tertiary, "Tertiary"),
        raster_to_points(waterlog, "Waterlog")
    ])

    gdf = gpd.GeoDataFrame(gdf, crs=crs)

    geojson_path = output_prefix + ".geojson"
    gdf.to_file(geojson_path, driver="GeoJSON")

    print("  Saved GeoJSON:", geojson_path)

    # ---- Visualization ----
    plt.figure(figsize=(10,8))

    dtm_norm = (dtm - np.nanmin(dtm)) / (np.nanmax(dtm) - np.nanmin(dtm))
    plt.imshow(dtm_norm, cmap='gray', alpha=0.5)

    primary_vis   = binary_dilation(primary, iterations=2)
    secondary_vis = binary_dilation(secondary, iterations=1)
    tertiary_vis  = binary_dilation(tertiary, iterations=1)

    overlay = np.zeros((*dtm.shape, 4))

    overlay[primary_vis]   = [0, 1, 1, 1]
    overlay[secondary_vis] = [1, 0.5, 0, 0.9]
    overlay[tertiary_vis]  = [0.3, 1, 0.3, 0.7]
    overlay[waterlog]      = [1, 0, 0, 0.8]

    plt.imshow(overlay)
    plt.title("Optimized Drainage Network")
    plt.axis('off')

    plot_path = output_prefix + ".png"
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.close()

    print("  Saved Plot:", plot_path)

In [20]:
import rasterio
import numpy as np
from pysheds.grid import Grid
import geopandas as gpd
from shapely.geometry import Point
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import binary_dilation

def process_dtm(dtm_path, output_prefix):
    print(f"\nProcessing: {os.path.basename(dtm_path)}")

    # ---- Load DTM ----
    with rasterio.open(dtm_path) as src:
        dtm = src.read(1)
        transform = src.transform
        crs = src.crs

    dtm = np.where(np.isnan(dtm), np.nanmean(dtm), dtm)

    # ---- Hydrology ----
    grid = Grid.from_raster(dtm_path)
    dem  = grid.read_raster(dtm_path)

    pit_filled = grid.fill_pits(dem)
    flooded    = grid.fill_depressions(pit_filled)
    inflated   = grid.resolve_flats(flooded)

    flow_dir = grid.flowdir(inflated)
    acc      = grid.accumulation(flow_dir)

    # ---- Thresholds (slightly relaxed for stability) ----
    primary_thresh   = np.percentile(acc, 98)
    secondary_thresh = np.percentile(acc, 95)
    tertiary_thresh  = np.percentile(acc, 90)

    primary   = acc >= primary_thresh
    secondary = (acc >= secondary_thresh) & (acc < primary_thresh)
    tertiary  = (acc >= tertiary_thresh) & (acc < secondary_thresh)

    # ---- Waterlogging ----
    gy, gx = np.gradient(dtm)
    slope  = np.sqrt(gx**2 + gy**2)

    low_slope = slope < np.percentile(slope, 25)
    high_acc  = acc > np.percentile(acc, 75)
    low_elev  = dtm < np.percentile(dtm, 30)

    waterlog = low_slope & high_acc & low_elev

    # ---- Debug stats ----
    print("  Pixel counts →",
          "Primary:", int(primary.sum()),
          "Secondary:", int(secondary.sum()),
          "Tertiary:", int(tertiary.sum()),
          "Waterlog:", int(waterlog.sum()))

    # ---- Safe raster → points ----
    def raster_to_points(mask, label):
        rows, cols = np.where(mask)

        if len(rows) == 0:
            return gpd.GeoDataFrame(
                {"geometry": [], "type": []},
                geometry="geometry",
                crs=crs
            )

        xs, ys = rasterio.transform.xy(transform, rows, cols)

        df = pd.DataFrame({
            "geometry": [Point(x, y) for x, y in zip(xs, ys)],
            "type": [label] * len(xs)
        })

        return gpd.GeoDataFrame(df, geometry="geometry", crs=crs)

    # ---- Create GeoDataFrame ----
    gdf_list = [
        raster_to_points(primary, "Primary"),
        raster_to_points(secondary, "Secondary"),
        raster_to_points(tertiary, "Tertiary"),
        raster_to_points(waterlog, "Waterlog")
    ]

    gdf = gpd.GeoDataFrame(
        pd.concat(gdf_list, ignore_index=True),
        geometry="geometry",
        crs=crs
    )

    # ---- Save GeoJSON ----
    geojson_path = output_prefix + ".geojson"
    gdf.to_file(geojson_path, driver="GeoJSON")
    print("  Saved GeoJSON:", geojson_path)

    # ---- Visualization ----
    plt.figure(figsize=(10, 8))

    dtm_norm = (dtm - np.nanmin(dtm)) / (np.nanmax(dtm) - np.nanmin(dtm))
    plt.imshow(dtm_norm, cmap='gray', alpha=0.5)

    primary_vis   = binary_dilation(primary, iterations=2)
    secondary_vis = binary_dilation(secondary, iterations=1)
    tertiary_vis  = binary_dilation(tertiary, iterations=1)

    overlay = np.zeros((*dtm.shape, 4))

    overlay[primary_vis]   = [0, 1, 1, 1]
    overlay[secondary_vis] = [1, 0.5, 0, 0.9]
    overlay[tertiary_vis]  = [0.3, 1, 0.3, 0.7]
    overlay[waterlog]      = [1, 0, 0, 0.8]

    plt.imshow(overlay)
    plt.title("Optimized Drainage Network")
    plt.axis('off')

    plot_path = output_prefix + ".png"
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.close()

    print("  Saved Plot:", plot_path)

In [21]:
for dtm_file in dtm_files:
    dtm_path = os.path.join(INPUT_DIR, dtm_file)

    name = os.path.splitext(dtm_file)[0]
    output_prefix = os.path.join(OUTPUT_DIR, name + "_drainage")

    process_dtm(dtm_path, output_prefix)

print("\n All DTMs processed successfully!")


Processing: 67169_5NKR_CHAKHIRASINGH_DTM.tif
  Pixel counts → Primary: 18205 Secondary: 27359 Tertiary: 48702 Waterlog: 0
  Saved GeoJSON: /content/drive/MyDrive/SVAMITVA_Drainage_Output/67169_5NKR_CHAKHIRASINGH_DTM_drainage.geojson
  Saved Plot: /content/drive/MyDrive/SVAMITVA_Drainage_Output/67169_5NKR_CHAKHIRASINGH_DTM_drainage.png

Processing: Dhal_Hoshiarpur_31235_DTM.tif
  Pixel counts → Primary: 7123 Secondary: 10721 Tertiary: 18333 Waterlog: 0
  Saved GeoJSON: /content/drive/MyDrive/SVAMITVA_Drainage_Output/Dhal_Hoshiarpur_31235_DTM_drainage.geojson
  Saved Plot: /content/drive/MyDrive/SVAMITVA_Drainage_Output/Dhal_Hoshiarpur_31235_DTM_drainage.png

Processing: KHAPRETA_510206_DTM.tif
  Pixel counts → Primary: 13355 Secondary: 20211 Tertiary: 34789 Waterlog: 0
  Saved GeoJSON: /content/drive/MyDrive/SVAMITVA_Drainage_Output/KHAPRETA_510206_DTM_drainage.geojson
  Saved Plot: /content/drive/MyDrive/SVAMITVA_Drainage_Output/KHAPRETA_510206_DTM_drainage.png

Processing: DHUNDA_FAT